# Experiment 9: Support Vector Machines (linear, RBF, polynomial)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.svm import SVC

X, y = make_classification(n_samples=200, n_features=2, n_redundant=0, n_informative=2,
                           n_clusters_per_class=1, class_sep=1.0, random_state=3)

def plot_svm(clf, X, y, title):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.2, cmap="coolwarm")
    plt.contour(xx, yy, Z, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k")
    plt.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
                s=150, facecolors="none", edgecolors="green", label="support vectors")
    plt.title(f"{title} (train acc = {clf.score(X, y):.3f})")
    plt.legend()

In [ ]:
plt.figure(figsize=(18, 5))
for i, (kernel, extra) in enumerate([("linear", {}), ("rbf", {"gamma": "scale"}),
                                     ("poly", {"degree": 3})], 1):
    plt.subplot(1, 3, i)
    clf = SVC(kernel=kernel, C=1.0, **extra).fit(X, y)
    plot_svm(clf, X, y, f"SVM {kernel}")
plt.show()

In [ ]:
# Tune C and gamma for an RBF SVM with grid search
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3, random_state=0)

pipe = make_pipeline(StandardScaler(), SVC(kernel="rbf"))
param_grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.001, 0.01, 0.1, 1]}
grid = GridSearchCV(pipe, param_grid, cv=5).fit(X_tr, y_tr)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 3))
print("Test accuracy:", round(grid.score(X_te, y_te), 3))